In [ ]:
# Imports + setup de MLflow — mlflow.set_tracking_uri, set_experiment, carga del CSV y definición del preprocessor.

import mlflow
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("churn-prediction")

df = pd.read_csv("../data/raw/churn.csv")

numeric_features = [
    'Call  Failure', 'Subscription  Length', 'Charge  Amount',
    'Seconds of Use', 'Frequency of use', 'Frequency of SMS',
    'Distinct Called Numbers', 'Age Group', 'Age', 'Customer Value',
]
categorical_features = ['Complains', 'Tariff Plan', 'Status']

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))]), categorical_features),
])

2026/08/30 00:46:02 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/30 00:46:02 INFO mlflow.store.db.utils: Updating database tables
2026/08/30 00:46:05 INFO mlflow.tracking.fluent: Experiment with name 'churn-prediction' does not exist. Creating a new experiment.


In [ ]:
# Run logreg_baseline — reproduce el modelo de 02_baseline.ipynb, pero ahora logueado en MLflow.

with mlflow.start_run(run_name="logreg_baseline"):
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(random_state=42, max_iter=1000)),
    ])
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_proba)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)

    mlflow.sklearn.log_model(
    pipeline,
    artifact_path="model",
    skops_trusted_types=["numpy.dtype"],
    )

    print(f"ROC-AUC: {roc_auc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

2026/08/30 00:54:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/30 00:55:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


ROC-AUC: 0.9208 | Precision: 0.8400 | Recall: 0.4242 | F1: 0.5638


In [ ]:
# Run random_forest_balanced — con class_weight="balanced".

from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="random_forest_balanced"):
    rf_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            class_weight="balanced",
            random_state=42,
        )),
    ])
    rf_pipeline.fit(X_train, y_train)

    y_pred = rf_pipeline.predict(X_test)
    y_proba = rf_pipeline.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_proba)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)

    mlflow.sklearn.log_model(rf_pipeline, name="model", skops_trusted_types=["numpy.dtype"])

    print(f"ROC-AUC: {roc_auc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

2026/08/30 00:58:07 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


ROC-AUC: 0.9848 | Precision: 0.7273 | Recall: 0.9697 | F1: 0.8312


In [ ]:
# Run xgboost_weighted — con scale_pos_weight.

from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

with mlflow.start_run(run_name="xgboost_weighted"):
    xgb_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.1,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=42,
        )),
    ])
    xgb_pipeline.fit(X_train, y_train)

    y_pred = xgb_pipeline.predict(X_test)
    y_proba = xgb_pipeline.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_proba)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_param("model_type", "XGBClassifier")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("scale_pos_weight", scale_pos_weight)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)

    mlflow.sklearn.log_model(
    xgb_pipeline,
    name="model",
    skops_trusted_types=["numpy.dtype", "xgboost.core.Booster", "xgboost.sklearn.XGBClassifier"],
    )

    print(f"ROC-AUC: {roc_auc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

2026/08/30 01:00:43 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


ROC-AUC: 0.9908 | Precision: 0.8455 | Recall: 0.9394 | F1: 0.8900


In [ ]:
# Tabla comparativa — mlflow.search_runs(...) para ver los 3 modelos lado a lado.

runs = mlflow.search_runs(order_by=["metrics.roc_auc DESC"])
runs[["run_id", "tags.mlflow.runName", "metrics.roc_auc", "metrics.precision", "metrics.recall", "metrics.f1"]]

,run_id,tags.mlflow.runName,metrics.roc_auc,metrics.precision,metrics.recall,metrics.f1
0,7f539438e21647dd8d80e747f5de10c6,xgboost_weighted,0.990755,0.845455,0.939394,0.889952
1,3680b1ce005e4ef28859f07c467dc7f6,xgboost_weighted,0.990755,0.845455,0.939394,0.889952
2,45e0823fe7ee4ec9a7b5286004741566,random_forest_balanced,0.984820,0.727273,0.969697,0.831169
3,da314ae2cf73450994a4934b2784276a,logreg_baseline,0.920752,0.840000,0.424242,0.563758
4,63e6ebda69754aa6aac93e64da275339,logreg_baseline,0.920752,0.840000,0.424242,0.563758


In [9]:
# Registro del modelo — mlflow.register_model(...) sobre el run de XGBoost.

model_uri = "runs:/7f539438e21647dd8d80e747f5de10c6/model"
registered_model = mlflow.register_model(model_uri=model_uri, name="churn-prediction-model")
print(registered_model.version)

Successfully registered model 'churn-prediction-model'.
2026/08/30 01:12:10 WARNING mlflow.tracking._model_registry.fluent: Run with id 7f539438e21647dd8d80e747f5de10c6 has no artifacts at artifact path 'model', registering model based on models:/m-8c1d1ac4044f442d8404781499b65c68 instead


1


Created version '1' of model 'churn-prediction-model'.


In [ ]:
# Alias champion — client.set_registered_model_alias(...)

from mlflow import MlflowClient

client = MlflowClient()
client.set_registered_model_alias(
    name="churn-prediction-model",
    alias="champion",
    version=registered_model.version,
)

In [ ]:
# Verificación de carga — mlflow.sklearn.load_model("models:/churn-prediction-model@champion").

loaded_model = mlflow.sklearn.load_model("models:/churn-prediction-model@champion")
print(loaded_model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Call  Failure',
                                                   'Subscription  Length',
                                                   'Charge  Amount',
                                                   'Seconds of Use',
                                                   'Frequency of use',
                                                   'Frequency of SMS',
                                                   'Distinct Called Numbers',
                                                   'Age Group', 